[\[GitHub\] Jupyter Notebook](https://github.com/sslastochkin/mcab/blob/main/docs/guide/02_load_your_data.ipynb)
  
# Loading Your Own Data  
  
This lesson will be lighter than the previous one.  
In it we will cover:  
* how to apply `mcab` to your own data
* what data classes exist in `mcab` and why they were introduced  
  
## What Advantages Does Using Real Data Have Over Synthetic Data?
  
You may have encountered approaches where simulations are run on a normal, exponential, or some other distribution.  
The authors of such methods assume that their data resembles some hypothetical distribution in shape and run simulations for that distribution, extrapolating the results to their actual data.  
This approach is less reliable and introduces additional risks; moreover, your data may be a mixture of several distributions or may only partially resemble a given one.  
A far more reliable approach is to use your own data, provided the sample size allows it.  
This is the philosophy of **mcab**.

## General Concepts
  
Before we begin, let us briefly outline what an A/B test simulation pipeline looks like.  
  
First, we load the data.  
Then:
1. add an effect to the test group (if needed)
2. apply linearization (if needed and if we have a ratio metric)
3. apply variance reduction (if needed, e.g. CUPED, etc.)
4. obtain the p-value
5. correct the p-value (if needed, e.g. Bonferroni correction)
6. aggregate the results
  
`mcab` provides a flexible high-level API where you can add and modify the required steps like building blocks, using a minimum amount of your own code.  
  
For clarity, let us visualize the simulation process on a diagram:

<img src="https://raw.githubusercontent.com/sslastochkin/mcab/main/docs/img/mcab_shema.png" width="300"/>

As the diagram shows, the simulation pipeline involves various optional steps.  
Whether a step can be applied at all — and how exactly it is applied — depends on the data we have.  
  
For example:
* linearization can only be applied to ratio metrics
* if our metric is user conversion, we cannot inject an effect such that the group mean falls below 0 or above 1
* and so on
  
`mcab` offers a high-level framework with a flexible API that lets the user ignore such nuances and get a ready-made, high-quality solution for simulating their experiment.  
The library asks the user to specify a minimal set of parameters upfront, with full flexibility to adjust them. The first such parameter is the input data type.

## Using Your Data  
  
`mcab` has 2 global classes for working with data:
- `AaDataIid` — for standard user-level data
- `AaDataRatio` — for ratio metrics

In [1]:
from mcab import (

    AaDataIid, # for simple user-level metrics
    AaDataRatio, # for ratio metrics

    # sandbox class we will look in this tutorial for some examples
    # but you dont need this in your simulations is `RandomData` class:
    RandomData,
)

In real-world data there are also 2 global classes:
1. User-level data (e.g. ARPU = average revenue per user)
2. Ratio metrics (e.g. average order value = revenue per user divided by number of purchases)
  
Each of these classes can be further split into 2 depending on whether the metric is a proportion.  
- User-level proportion data: user conversion to some action — either 0 (did not convert) or 1 (converted). For example, whether the user renewed a subscription.
- Ratio proportion metrics: there is a numerator and a denominator, but the numerator cannot exceed the denominator (e.g. CTR).  
  
For learning purposes we will simply generate these 4 data types, but you can also load any of them via `pd.read_csv` or any other convenient method from your existing data:

In [2]:
rd = RandomData(42)

arpu_raw       = rd.exponential_data()
conversion_raw = rd.proportion_data()
ctr_raw        = rd.ratio_data_ctr()
avg_bill_raw   = rd.ratio_data_avg_bill()

print('Iid Data:')
print(f'arpu_raw:       {type(arpu_raw)} shape: {arpu_raw.shape}')
print(f'conversion_raw: {type(conversion_raw)} shape: {conversion_raw.shape}')
print('')
print('Ratio Data:')
print(f'ctr_raw:      {type(ctr_raw)} shape: {ctr_raw[0].shape, ctr_raw[1].shape}')
print(f'avg_bill_raw: {type(avg_bill_raw)} shape: {avg_bill_raw[0].shape, avg_bill_raw[1].shape}')

Iid Data:
arpu_raw:       <class 'numpy.ndarray'> shape: (10000,)
conversion_raw: <class 'numpy.ndarray'> shape: (10000,)

Ratio Data:
ctr_raw:      <class 'tuple'> shape: ((10000,), (10000,))
avg_bill_raw: <class 'tuple'> shape: ((10000,), (10000,))


Let us look at user-level data (Iid Data) in more detail.  
It can be represented either as a one-dimensional `numpy.array` or as a `pd.Series`.
- `arpu_raw` — revenue per user data.  
   Each row = a unique user.  
   Value = total revenue for that user.

In [3]:
arpu_raw

array([2404.20860397, 2336.18965582, 2384.76099987, ...,  539.9254313 ,
       2483.04487108,   21.95119149])

- `conversion_raw` — per-user conversion data (converted=1, not converted=0).  
   Each row = a unique user, values are either 0 or 1.

In [4]:
conversion_raw

array([1, 0, 1, ..., 0, 1, 1])

Now let us talk about ratio metrics.  
While iid metrics could be represented with a one-dimensional `numpy.array` or `pd.Series`,  
a ratio metric requires a tuple of two elements because it has a numerator and a denominator.  
  
The numerator and denominator can each be represented the same way as iid metrics (a one-dimensional array).  
  
Let us look at each metric in more detail:
- `avg_bill_raw` — average order value per user data. Numerator = total revenue per user, denominator = number of purchases per user.

In [5]:
avg_bill_raw

(array([3036.43846203, 4736.7308586 , 1695.82574228, ...,  548.69201029,
         658.56014132,  129.51843694]),
 array([9, 3, 4, ..., 3, 1, 4]))

- `ctr_raw` — per-user CTR data (ratio of clicks to impressions).  
   Unlike average order value, the numerator is bounded from above — it cannot exceed the denominator.

In [6]:
ctr_raw

(array([3, 1, 1, ..., 0, 0, 0]), array([15, 13, 10, ...,  8,  4,  7]))

### Loading Data into the mcab Format  
  
Simply create an instance of one of the two classes and pass the `proportion=True` flag if needed:

In [7]:
arpu       = AaDataIid(arpu_raw)
conversion = AaDataIid(conversion_raw, proportion=True)

avg_bill = AaDataRatio(avg_bill_raw)
ctr      = AaDataRatio(ctr_raw, proportion=True)

We will pass these class instances as variables to subsequent `mcab` methods.  
  
## Lesson Summary  
  
In this lesson we discussed what data types `mcab` supports and how you can load them into a format it understands.  
See you in the next lessons!